# Model Training

Load features & labels

In [11]:
import joblib
import pandas as pd
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

X_train_vec = joblib.load("../data/processed/X_train_vec.pkl")
X_test_vec = joblib.load("../data/processed/X_test_vec.pkl")
y_train = joblib.load("../data/processed/y_train.pkl")
y_test = joblib.load("../data/processed/y_test.pkl")

print(f"X_train_vec shape: {X_train_vec.shape}")
print(f"X_test_vec shape:  {X_test_vec.shape}")
print(f"y_train size: {len(y_train)}")
print(f"y_test size:  {len(y_test)}")


X_train_vec shape: (4135, 7591)
X_test_vec shape:  (1034, 7591)
y_train size: 4135
y_test size:  1034


Multinomial naive bayes

In [12]:
nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)

y_pred_nb = nb_model.predict(X_test_vec)

nb_accuracy = accuracy_score(y_test, y_pred_nb)
nb_precision = precision_score(y_test, y_pred_nb, pos_label="spam")
nb_recall = recall_score(y_test, y_pred_nb, pos_label="spam")
nb_f1 = f1_score(y_test, y_pred_nb, pos_label="spam")

print("--- Multinomial Naive Bayes ---")
print(f"Accuracy:  {nb_accuracy:.4f}")
print(f"Precision: {nb_precision:.4f}")
print(f"Recall:    {nb_recall:.4f}")
print(f"F1-Score:  {nb_f1:.4f}")

print("\nConfusion Matrix (rows=actual, cols=predicted, order=[ham, spam]):")
print(confusion_matrix(y_test, y_pred_nb, labels=["ham", "spam"]))


--- Multinomial Naive Bayes ---
Accuracy:  0.9865
Precision: 0.9756
Recall:    0.9160
F1-Score:  0.9449

Confusion Matrix (rows=actual, cols=predicted, order=[ham, spam]):
[[900   3]
 [ 11 120]]


Logistic regression

In [13]:
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_vec, y_train)

y_pred_lr = lr_model.predict(X_test_vec)

lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr, pos_label="spam")
lr_recall = recall_score(y_test, y_pred_lr, pos_label="spam")
lr_f1 = f1_score(y_test, y_pred_lr, pos_label="spam")

print("--- Logistic Regression ---")
print(f"Accuracy:  {lr_accuracy:.4f}")
print(f"Precision: {lr_precision:.4f}")
print(f"Recall:    {lr_recall:.4f}")
print(f"F1-Score:  {lr_f1:.4f}")

print("\nConfusion Matrix (rows=actual, cols=predicted, order=[ham, spam]):")
print(confusion_matrix(y_test, y_pred_lr, labels=["ham", "spam"]))


--- Logistic Regression ---
Accuracy:  0.9797
Precision: 0.9911
Recall:    0.8473
F1-Score:  0.9136

Confusion Matrix (rows=actual, cols=predicted, order=[ham, spam]):
[[902   1]
 [ 20 111]]


SVM Model


In [14]:
from sklearn.svm import SVC
svm_model = SVC(kernel="linear", probability=True)

svm_model.fit(X_train_vec, y_train)

y_pred_svm = svm_model.predict(X_test_vec)
svm_accuracy = accuracy_score(y_test, y_pred_svm)

svm_precision = precision_score(
    y_test,
    y_pred_svm,
    pos_label="spam"
)

svm_recall = recall_score(
    y_test,
    y_pred_svm,
    pos_label="spam"
)

svm_f1 = f1_score(
    y_test,
    y_pred_svm,
    pos_label="spam"
)
print("--- SVM ---")
print(f"Accuracy:  {svm_accuracy:.4f}")
print(f"Precision: {svm_precision:.4f}")
print(f"Recall:    {svm_recall:.4f}")
print(f"F1-Score:  {svm_f1:.4f}")
print("\nConfusion Matrix (rows=actual, cols=predicted, order=[ham, spam]):")
print(confusion_matrix(
    y_test,
    y_pred_svm,
    labels=["ham", "spam"]
))

c:\Users\DE\Desktop\ML_Assignment\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


--- SVM ---
Accuracy:  0.9845
Precision: 1.0000
Recall:    0.8779
F1-Score:  0.9350

Confusion Matrix (rows=actual, cols=predicted, order=[ham, spam]):
[[903   0]
 [ 16 115]]


Results Table

In [15]:
results = pd.DataFrame({
    "Model": [
        "Naive Bayes",
        "Logistic Regression",
        "SVM"
    ],
    "Accuracy": [
        nb_accuracy,
        lr_accuracy,
        svm_accuracy
    ],
    "Precision": [
        nb_precision,
        lr_precision,
        svm_precision
    ],
    "Recall": [
        nb_recall,
        lr_recall,
        svm_recall
    ],
    "F1": [
        nb_f1,
        lr_f1,
        svm_f1
    ]
})
print("--- Baseline Models Comparison ---")
print(results)


--- Baseline Models Comparison ---
                 Model  Accuracy  Precision    Recall        F1
0          Naive Bayes  0.986460   0.975610  0.916031  0.944882
1  Logistic Regression  0.979691   0.991071  0.847328  0.913580
2                  SVM  0.984526   1.000000  0.877863  0.934959


In [17]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, make_scorer

# 1. Create a custom scorer specifying the positive label
f1_scorer = make_scorer(f1_score, pos_label="spam")

# 2. Define the parameters to test
params = {
    "C": [0.01, 0.1, 1, 10, 100],
    "max_iter": [1000, 2000, 3000]
}

# 3. Pass the custom scorer to GridSearchCV
grid = GridSearchCV(
    LinearSVC(random_state=42),
    params,
    cv=5,
    scoring=f1_scorer  # <-- Custom scorer applied here
)

grid.fit(X_train_vec, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV Score (F1):", grid.best_score_)

c:\Users\DE\Desktop\ML_Assignment\venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\DE\Desktop\ML_Assignment\venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\DE\Desktop\ML_Assignment\venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\DE\Desktop\ML_Assignment\venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\DE\Desktop\ML_Assignment\venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\DE\Desktop\ML_Assignment\venv\Lib\site-packages\sklearn\svm\_

Best Parameters: {'C': 1, 'max_iter': 1000}
Best CV Score (F1): 0.9239426187202083


c:\Users\DE\Desktop\ML_Assignment\venv\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [22]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# 1. Train the final tuned model
best_svm = LinearSVC(C=1, max_iter=1000, random_state=42)
best_svm.fit(X_train_vec, y_train)

# 2. Predict on the unseen test set
y_pred_tuned = best_svm.predict(X_test_vec)

# 3. Calculate final metrics
print("--- Tuned SVM Test Performance ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_tuned):.6f}")
print(f"Precision: {precision_score(y_test, y_pred_tuned, pos_label='spam'):.6f}")
print(f"Recall:    {recall_score(y_test, y_pred_tuned, pos_label='spam'):.6f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_tuned, pos_label='spam'):.6f}")

# 4. Generate Confusion Matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred_tuned, labels=['ham', 'spam'])
print(cm)



--- Tuned SVM Test Performance ---
Accuracy:  0.984526
Precision: 1.000000
Recall:    0.877863
F1 Score:  0.934959

Confusion Matrix:
[[903   0]
 [ 16 115]]


Save trained models & results

In [21]:
import os

os.makedirs("../models", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

joblib.dump(nb_model, "../models/naive_bayes_model.pkl")
joblib.dump(lr_model, "../models/logistic_regression_model.pkl")
joblib.dump(svm_model, "../models/svm_model.pkl")
results.to_csv("../data/processed/model_results.csv", index=False)

print("Saved: ../models/naive_bayes_model.pkl")
print("Saved: ../models/logistic_regression_model.pkl")
print("Saved: ../data/processed/model_results.csv")



Saved: ../models/naive_bayes_model.pkl
Saved: ../models/logistic_regression_model.pkl
Saved: ../data/processed/model_results.csv
